In [1]:
import os
import pandas as pd
import torch
from PIL import Image
from torch.backends import cudnn
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import VisionDataset
from torchvision.transforms import InterpolationMode, v2
import torchvision.models as models



In [2]:
torch.manual_seed(3407)
torch.cuda.manual_seed(3407)

cudnn.deterministic = False
cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# paths
train_dir = "/kaggle/input/cassava-leaf-disease-classification/train_images/"
train_csv_path = "/kaggle/input/cassava-leaf-disease-classification/train.csv"
test_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images/"

# hyper‑parameters
model_a_img_size = 384
model_b_img_size = 384
batch_size = 16
num_workers = 4
num_classes = 5
tta = True  # enable a little test‑time augmentation




Using device: cuda


In [3]:
def load_or_fallback(path, fallback_fn):
    """
    Try to load a torch model from `path`. If it fails, create a model
    using `fallback_fn`, adapt its final layer to `num_classes`, and return it.
    """
    try:
        model = torch.load(path, map_location=device)
        print(f"Loaded model from {path}")
    except Exception as e:
        print(f"Could not load {path} ({e}), using fallback model.")
        model = fallback_fn()
        if hasattr(model, "fc"):  # ResNet style
            in_features = model.fc.in_features
            model.fc = torch.nn.Linear(in_features, num_classes)
        elif hasattr(model, "classifier"):  # EfficientNet / others
            if isinstance(model.classifier, torch.nn.Linear):
                in_features = model.classifier.in_features
                model.classifier = torch.nn.Linear(in_features, num_classes)
            else:
                model.classifier = torch.nn.Sequential(
                    torch.nn.Flatten(),
                    torch.nn.Linear(model.classifier[0].in_features, num_classes),
                )
        else:
            model = torch.nn.Sequential(
                model,
                torch.nn.Flatten(),
                torch.nn.Linear(1000, num_classes),
            )
    model.to(device)
    model.eval()
    return model


fallback_resnet18 = lambda: models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

model_a = load_or_fallback("/kaggle/input/vit-v1/vit_v1.pt", fallback_resnet18)
model_b = load_or_fallback(
    "/kaggle/input/vit-boosted/vit_boosted.pt", fallback_resnet18
)




Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


Could not load /kaggle/input/vit-v1/vit_v1.pt ([Errno 2] No such file or directory: '/kaggle/input/vit-v1/vit_v1.pt'), using fallback model.


100%|██████████| 44.7M/44.7M [00:00<00:00, 114MB/s]


Could not load /kaggle/input/vit-boosted/vit_boosted.pt ([Errno 2] No such file or directory: '/kaggle/input/vit-boosted/vit_boosted.pt'), using fallback model.


In [4]:
class TrainCassavaDataset(Dataset):
    """Dataset for training that returns image tensor and integer label."""

    def __init__(self, images_dir, csv_path, img_size, transform=None):
        self.images_dir = images_dir
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.resize = v2.Resize(
            (img_size, img_size), interpolation=InterpolationMode.BICUBIC
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row["image_id"])
        img = Image.open(img_path).convert("RGB")
        img = self.resize(img)
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label


# simple augmentations for a few epochs
train_transforms = v2.Compose(
    [
        v2.RandomResizedCrop((model_a_img_size, model_a_img_size), scale=(0.8, 1.0)),
        v2.RandomHorizontalFlip(),
        v2.ToTensor(),
        v2.ConvertImageDtype(torch.float32),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = TrainCassavaDataset(
    train_dir, train_csv_path, model_a_img_size, transform=train_transforms
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

# fine‑tune the fallback ResNet‑18 (model_a) for a couple of epochs
model_a.train()
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_a.parameters(), lr=1e-4)

epochs = 2
for epoch in range(epochs):
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model_a(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{epochs}] loss: {epoch_loss:.4f}")

model_a.eval()  # set back to evaluation mode




/usr/local/lib/python3.11/dist-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Epoch [1/2] loss: 0.5244
Epoch [2/2] loss: 0.3769


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [5]:
class CassavaDataset(VisionDataset):
    """Custom dataset for the Cassava test data."""

    def __init__(
        self,
        data_dir,
        model_a_size,
        model_b_size,
        transform=None,
        ttas=None,
    ):
        super().__init__(root=data_dir)
        self.transform = transform
        self.images = sorted(
            [
                f
                for f in os.listdir(data_dir)
                if os.path.isfile(os.path.join(data_dir, f))
                and f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]
        )
        self.ttas = ttas
        self.cc = v2.CenterCrop((600, 600))
        self.resize_model_a = v2.Resize(
            (model_a_size, model_a_size), interpolation=InterpolationMode.BICUBIC
        )
        self.resize_model_b = v2.Resize(
            (model_b_size, model_b_size), interpolation=InterpolationMode.BICUBIC
        )

    def __getitem__(self, idx):
        filename = self.images[idx]
        img_path = os.path.join(self.root, filename)
        img = Image.open(img_path).convert("RGB")
        img = self.cc(img)
        model_a_img = self.resize_model_a(img)
        model_b_img = self.resize_model_b(img)

        if self.ttas is not None and self.transform is not None:
            model_a_img = [self.transform(t(model_a_img)) for t in self.ttas]
            model_b_img = [self.transform(t(model_b_img)) for t in self.ttas]
        elif self.transform:
            model_a_img = self.transform(model_a_img)
            model_b_img = self.transform(model_b_img)

        return model_a_img, model_b_img, filename

    def __len__(self):
        return len(self.images)




In [6]:
test_transforms = v2.Compose(
    [
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

ttas = (
    [v2.RandomRotation(180), v2.RandomVerticalFlip(1), v2.RandomPerspective(p=1)]
    if tta
    else None
)

test_dataset = CassavaDataset(
    test_dir, model_a_img_size, model_b_img_size, transform=test_transforms, ttas=ttas
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,  # keep deterministic order
    num_workers=num_workers,
    pin_memory=True,
)

normalizer = torch.nn.Softmax(dim=1)



In [7]:
all_names = []
all_preds = []

with torch.no_grad():
    for batch_idx, (model_a_inputs, model_b_inputs, filenames) in enumerate(
        test_loader
    ):
        batch_sz = len(filenames)

        if tta:
            model_a_inputs = torch.cat(model_a_inputs, dim=0).to(device)
            model_b_inputs = torch.cat(model_b_inputs, dim=0).to(device)

            model_a_outputs = model_a(model_a_inputs)
            model_b_outputs = model_b(model_b_inputs)

            model_a_logits = torch.stack(
                torch.split(model_a_outputs, batch_sz), dim=0
            ).mean(dim=0)
            model_b_logits = torch.stack(
                torch.split(model_b_outputs, batch_sz), dim=0
            ).mean(dim=0)

            outputs = (0.9 * model_a_logits + 0.1 * model_b_logits) / 2.0
        else:
            model_a_inputs = model_a_inputs.to(device)
            model_b_inputs = model_b_inputs.to(device)

            model_a_outputs = model_a(model_a_inputs)
            model_b_outputs = model_b(model_b_inputs)

            outputs = (0.92 * model_a_outputs + 0.08 * model_b_outputs) / 2.0

        probs = normalizer(outputs)
        pred_labels = torch.argmax(probs, dim=1).cpu().tolist()

        all_names.extend(filenames)
        all_preds.extend(pred_labels)



In [8]:
submission_path = "submission.csv"
my_submission = pd.DataFrame({"image_id": all_names, "label": all_preds})
my_submission.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")
my_submission.head()

Submission written to submission.csv


,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,1
3,1234555380.jpg,1
4,1234571117.jpg,3
